In [ ]:
from pathlib import Path
from ocularrigidity.consts import ROOT_DATA_SMB
import numpy as np


# IMPORTANT: compression reading and loading use FFMPEG for efficiency! You do need it, and the default
# values currently hardcode my own path to ffmpeg.
from ocularrigidity.data.compression import mp4_to_cube, read_gray


from ocularrigidity.data.io import load_cube

# This is my own video
path_data = Path(
    "F:/NASA_Spectralis_Pulse/videoFolder/1"
)  # Path to the folder containing the cube.bin.

# Note: you can either use a localpath or link to a path mounted with SMB.
# In the latter case, make sure you did mount the SMB share.

# data = load_cube(path_data)

path_to_ffmpeg = "C:/ProgramData/chocolatey/lib/ffmpeg/tools/ffmpeg/bin/ffmpeg.exe"  # Path to ffmpeg, which you need to have installed on your system. Y
# If you have a video (mp4):
data = read_gray(
    path_data / "compressedOCT_woOutliers_median_threshold.mj2"
)  # This time, you need to provide the full path to the video, not just the folder.
# If you have a video (.mkv):
# data = read_gray(path_data / "video.mkv", ffmpeg=path_to_ffmpeg)


W0616 17:54:05.698000 386228 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Segmentation


In [2]:
from ocularrigidity.data.io import save_mask
from ocularrigidity.segmentation.utils import get_choroid_segmentation_model
from ocularrigidity.segmentation.inference import infer

model = (
    get_choroid_segmentation_model()
)  # Model is automatically downloaded on first call.

# See the docstring of the infer function for more details on the parameters. The most important ones are scale_factor, which controls the resizing of the input, which controls the pixel size, and batch_size, which controls the speed of inference (the larger, the faster, but also the more VRAM you need).
# The model was trained with a pixel size, axial of 1.95um and lateral of 5.95um (at a resolution of 1536x1024). Match to your data.
mask = infer(model, data, scale_factor=(2), batch_size=16, device="cuda:0")
# Save the masks as a numpy array for later use:
OUTPUT_MASK_PATH = Path("F:/NASA_Spectralis_Pulse/maskFolder/1/mask.npz")
save_mask(mask, OUTPUT_MASK_PATH)  # Use packed + zstd compressed, so very light

c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: DiceScore metric currently defaults to `average=micro`, but will change to`average=macro` in the v1.9 release. If you've explicitly set this parameter, you can ignore this warning.
  warnings.warn(*args, **kwargs)


# Registration from segmentation


In [ ]:
from ocularrigidity.registration.registration_engine import (
    VideoRegistrator,
    RegistrationTransform,
)

ROOT_DATA = "F:/NASA_Spectralis_Pulse/videoFolder"
registrator = VideoRegistrator(
    video=Path(
        "1/compressedOCT_woOutliers_median_threshold.mj2"
    ),  # Relative to root_data, either a filepath or a folderpath. If a folder, the code will look for a cube.bin.
    root_data=Path(ROOT_DATA),
    root_masks=Path("F:/NASA_Spectralis_Pulse/maskFolder"),
    skip_first_n_frames=5,  # Whether to skip the first frames
    drop_last_n_frames=5,  # Whether to drop the last frames
    refine_iters=1,  # Not necessary
    transform=RegistrationTransform.TILT,
    flatten=False,  # Whether to flatten the Bruch's membrane boundary.
    horizontal_scaling=False,  #
    horizontal_alignment=False,
    verbose=True,
    use_encoded_video=True,  # True or False, depending on whether you feed the path to the video as mp4/mkv or as the cube.bin
)

# This object will perform the registration lazily and cache the results, so you can call the properties in any order and they will be computed only once. The most important ones are registered_frames, which gives you the registered video, and thickness, which gives you the registered thickness of the choroid.

registered_frames = registrator.registered_frames  # Registered video frames, as a numpy array of shape (T, H, W). This is a property, so the registration is performed lazily when you call it. Note that the registration is performed on the fly, so it may take some time to compute the registered frames, but they will be cached for later use, so you can call this property multiple times without recomputing the registration.

registered_masks = registrator.registered_masks  # Registered masks, as a numpy array of shape (T, H, W). This is a property, so the registration is performed lazily when you call it.

thickness = registrator.thickness  # Registered thickness, as a numpy array of shape (T, H). This is a property, so the registration is performed lazily when you call it.

# To get only the registered BM, CSI
lines = registrator.registered_lines

# Whole cardiac pipeline


In [ ]:
from ocularrigidity.data.compression import cube_to_mkv_lossless, cube_to_mp4
from ocularrigidity.motion.pulsation import CardiacCycleExtractor
from ocularrigidity.motion.results import CardiacPipelineResults


cardiac_cycle = CardiacCycleExtractor(
    registrator,
    timestamps_path="F:/NASA_Spectralis_Pulse/videoFolder/1/timestamps.txt",  # Path to the timestamps of the video, which should a text file containing the timestamps. Note that the sampling rate is estimated from the timestamps, so it is important to have them.
    bpm_range=(
        40,
        140,
    ),  # Range of expected heart rates, in bpm. This is used to filter the frequencies of the pulsation signal.
    butter_order=4,  # Not used anymore
    override_bpm=None,  # If you want to override the estimated bpm, you can provide it here. This is useful if the estimation fails, which can happen if the video is too short or too noisy.
    expected_bpm=60,  # Expected bpm, used for the estimation of the cardiac cycle. This is used to estimate the quality of the extracted cardiac cycle, so it is important to provide a reasonable value.
    skip_first_n_frames=5,  # Whether to skip the first frames, which can be noisy. Should match the value used for the registration.
    drop_last_n_frames=5,  # Whether to drop the last frames, which can be noisy. Should match the value used for the registration.
    sigma_col=5,  # Lateral smoothing of the thickness, in columns (pixel unit)
    col_slice=None,  # If you want to restrict the analysis to a subset of columns, you can provide a slice here. This is useful if you have a lot of lateral motion, which can make the registration and pulsation extraction less reliable. Should be a slice object, e.g. slice(100, 1000) to restrict to columns 100 to 1000.
    n_separable_components=16,  # Instead of analysing the signal in each column independently, which can be noisy, we can perform a PCA/ICA and analyse the first n_separable_components components, which are spatially smoothed and therefore more robust. This is especially useful if you have a lot of lateral motion, which can make the registration and pulsation extraction less reliable.
)

results = CardiacPipelineResults.from_extractor(
    cardiac_cycle
)  # Pickable object containing all the results of the cardiac cycle extraction, including the extracted cardiac cycle, the estimated bpm, the quality of the extraction, etc. You can also save it to a file and load it later. You can also find the actual/interpolated/filtered thicknees.

results.save(
    "F:/NASA_Spectralis_Pulse/videoFolder/1/cardiac_results.pkl"
)  # Save the results to a file, so you can load it later without having to recompute everything.

loaded_results = CardiacPipelineResults.load(
    "F:/NASA_Spectralis_Pulse/videoFolder/1/cardiac_results.pkl"
)  # Load the results from a file.

# Until here, the actual frames are never used, so it's very fast. The slow part if the computation of the n-cycle video, which requires to go back to the registered frames (so compute them) and fold them according to their phase in the cardiac cycle. You can control the trade-off between temporal resolution and noise with the parameters of the compute_n_cycle_video function, which are explained in the docstring of the function.
cardiac_cycle.compute_n_cycle_video(
    n_bins=30,  # Number of bins to use for the n-cycle video. The frames are assigned to the bins according to their phase in the cardiac cycle, so the more bins, the better the temporal resolution, but also the noisier the n-cycle video.
    n_cycle=3,
    target_frames_per_bin=25,  # Target number of frames per bin, which controls the trade-off between temporal resolution and noise. The more frames per bin, the less noisy the n-cycle video, but also the less temporal resolution.
    fold_method="median",  # Method to use for folding the frames into the n-cycle video. Can be "median", "mean"
    phase_method="iq",  # Method to use for estimating the phase of each frame in the cardiac cycle. Can be "iq" for Hilbert transform, "peak_locked" for peak detection.
)

cycle = cardiac_cycle.cycles  # The actual N-cycle video, as a numpy array of shape (n_bins, H, W). The frames are assigned to the bins according to their phase in the cardiac cycle.
counts = cardiac_cycle.counts  # The number of frames in each bin, as a numpy array of shape (n_bins,). This is useful to know how many frames are in each bin, and therefore how noisy the n-cycle video is.

# You can save the n-cycle videos as:
cube_to_mkv_lossless(
    cycle,
    "F:/NASA_Spectralis_Pulse/videoFolder/1/n_cycle_video.mkv",
    ffmpeg=path_to_ffmpeg,
    fps=30,
)  # This will save the n-cycle video as a lossless mkv, which can be opened with any video player.

Selected IC 2: BPM = 98.3, FAP = 2.54e-06, concentration = 0.16
Fold: counts min/mean/max = 1/4.9/8
Fold: counts min/mean/max = 1/4.0/8
Fold: counts min/mean/max = 1/3.8/6


In [ ]:
# You can estimate the amplitude of the pulsation in the one-cycle
from ocularrigidity.motion.displacement import (
    compute_delta_A_from_displacements,
    compute_delta_A_from_displacements,
    extract_displacement_at_boundaries,
)
from ocularrigidity.segmentation.closing_structures import trim_choroid
from ocularrigidity.viewer.gif import render_mask_quiver


masks_one_cycle = infer(
    model.cuda(),
    cycle,
    batch_size=8,
    scale_factor=(1.0, 1.0),
    return_logit=False,
    use_graphcut=False,
    use_amp=True,
    device="cuda:0",
    verbose=True,
    graphcut_kwargs=dict(
        temporal_smooth=False,
        temporal_iterations=4,
        temporal_mu=1.0,
        temporal_sigma=2.0,
        lambda_smooth=1.0,
    ),
)
trimmed_masks = trim_choroid(
    masks_one_cycle,
    75,
)  # To avoid boundary issues in the computation of the displacement, we trim the choroid on each side.

# We now compute the pixel displacement at the boundary of the choroid. To do so, the following function uses demons (or optical flow, depends on my mood)
displacements, reference_border_coords = extract_displacement_at_boundaries(
    cycle[
        0:30
    ],  # Only the first cycle, with corresponds to the first 30 frames (because n_bins=30)
    trimmed_masks[0:30],
    smooth_window=15,  # savgol filter to smooth out the displacement signal, which can be noisy. The window size is in frames, so 15 means that the displacement at each frame is computed as the median of the displacements in a window of 15 frames around it. You can adjust this parameter to control the trade-off between temporal resolution and noise in the displacement signal.
)
#
delta_a_differential = compute_delta_A_from_displacements(
    reference_border_coords, displacements
)  # We now have the variation of the area of the choroid over time, which is directly related to the pulsation. You can then analyse this signal to extract the amplitude and other characteristics of the pulsation.

# This will render a nice quiver gif to visualize the displacement of the choroid over one-cardiac cycle. You can adjust the parameters to control the appearance of the quiver plot, such as the stride, which controls the density of the arrows, and the arrow_scale, which controls the size of the arrows.
render_mask_quiver(
    (cycle[0:30] * 255).astype(np.uint8),
    trimmed_masks[0:30],
    "test_deltaA_opticalFlow.gif",
    side_by_side=False,
    smooth_window=11,
    stride=4,  # Stride to control the density of the arrows. A stride of 1 means that there is an arrow for each pixel, a stride of 2 means that there is an arrow for every 2 pixels, etc. You can adjust this parameter to control the density of the arrows in the quiver plot.
    arrow_cmap="RdYlGn_r",
    arrow_scale=30,  # Multiplier for the size of the arrows.
    fps=30,
    lk_window=35,  # Parameters for the Lucas-Kanade optical flow, which is used to compute the displacement between the frames. A larger windows is better to visualize "global" motion.
    cyclic=True,
)

Inference: 100%|██████████| 12/12 [00:01<00:00,  7.42it/s]


In [ ]:
render_mask_quiver(
    (mask[0:30] * 255).astype(np.uint8),
    trimmed_masks[0:30],
    "F:/NASA_Spectralis_Pulse/videoFolder/1/test_mask.gif",
    side_by_side=False,
    smooth_window=11,
    stride=4,  # Stride to control the density of the arrows. A stride of 1 means that there is an arrow for each pixel, a stride of 2 means that there is an arrow for every 2 pixels, etc. You can adjust this parameter to control the density of the arrows in the quiver plot.
    arrow_cmap="RdYlGn_r",
    arrow_scale=10,  # Multiplier for the size of the arrows.
    fps=30,
    lk_window=35,  # Parameters for the Lucas-Kanade optical flow, which is used to compute the displacement between the frames. A larger windows is better to visualize "global" motion.
    cyclic=True,
)